In [1]:
# This document contains useful functions for experimentation with caterpillars, as well as computations
# that we are currently exploring

In [2]:
%run DNCfundamentals.ipynb
%run helperFunctions.ipynb

In [3]:
# Ouptut: list containing all caterpillars on n vertices in the form of compositions
def generate_caterpillars(n):
    seen = set()
    compositions = list(Compositions(n))
    caterpillar_list = []
    
    for comp in compositions:
        if comp[0] != 1 and comp[-1] != 1:
            reversal = comp[::-1]
            if tuple(reversal) not in seen:
                seen.add(tuple(comp))
                caterpillar_list.append(comp)
    
    return caterpillar_list

In [4]:
# Output: list of all proper caterpillars on n vertices in the form of compositions
def generate_proper_caterpillars(n):
    seen = set()
    compos = list(Compositions(n))
    caterpillar_list = []
    proper_compos = []
    
    for comp in compos:
        if count_non_ones(comp) == len(comp):
            proper_compos.append(comp)
    
    for comp in proper_compos:
        if comp[0] != 1 and comp[-1] != 1:
            reversal = comp[::-1]
            if tuple(reversal) not in seen:
                seen.add(tuple(comp))
                caterpillar_list.append(comp)
    
    return caterpillar_list

In [5]:
# Function that, given a CSF vector, returns an array containing the terms in the CSF 
#     corresponding to partitions of length 1 and 2 less than the leading in the form (mu, c_mu)

def mu_nu_partitions(CSFvector,n):
    partitions = Partitions(n).list()
    leading = get_leading_partition(CSFvector, n)
    mu_nu_partitions = []
    
    for i in range(len(partitions)):
        lbda = partitions[i]
        if len(lbda) == len(leading)-1 and count_ones(lbda)==0 and CSFvector[i]!=0:
            lst = [lbda, CSFvector[i]]
            tpl = tuple(lst)
            mu_nu_partitions.append(tpl)
    
    for i in range(len(partitions)):
        lbda = partitions[i]
        if len(lbda) == len(leading)-2 and count_ones(lbda)==0 and CSFvector[i]!=0:
            lst = [lbda, CSFvector[i]]
            tpl = tuple(lst)
            mu_nu_partitions.append(tpl)
    return mu_nu_partitions

In [6]:
# Output: a map that sends a caterpillar --> a tuple containing its leading partition, and its mu and nu partitions
#         as defined in the function above
def caterpillars_leading_nu_map(n):
    map = {}
    for comp in generate_caterpillars(n):
        if count_ones(comp)==1:
            C=create_caterpillar(comp)
            C=C.copy()
            lst = list(mu_nu_partitions(CSF_tree(C), n))
            lst.insert(0, tuple(sorted(comp, reverse=True)))
            tpl = tuple(lst)
            map[tuple(comp)]=tpl
    return map

# Input: a composition (comp) representing a caterpillars
# Outputs a dictionary containing edges with multiplicity appearing in a caterpillar with composition comp
def caterpillar_edges(comp):
    edge_map = {}
    for i in range(len(comp)-1):
        lst = sorted([comp[i], comp[i+1]], reverse=True)
        edge = tuple(lst)
        if edge in edge_map.keys():
            curr = edge_map[edge]
            edge_map[edge] = curr+1
        else:
            edge_map[edge]=1
    return edge_map

In [35]:
n=15

mp = caterpillars_leading_nu_map(n)

flipped = {}

for key, value in mp.items():
    tplkey = tuple(key)
    tplvalue=tuple(value)
    
    if tplvalue not in flipped.keys():
        flipped[tplvalue] = tuple([tplkey])

    else:
        lst = list(flipped[tplvalue])
#         print(lst)
        lst.append(tplkey)
        tpl = tuple(lst)
        flipped[tplvalue]=tpl

for key in flipped.keys():
    if len(flipped[key])>1:
        for i in range(len(flipped[key])-1):
            if not caterpillar_edges(flipped[key][i]) == caterpillar_edges(flipped[key][i+1]):
                print(key, '-->', flipped[key])
                break

((3, 3, 2, 2, 2, 2, 1), ([3, 3, 3, 2, 2, 2], 2), ([6, 3, 2, 2, 2], 2), ([5, 3, 3, 2, 2], 5), ([4, 3, 3, 3, 2], 2)) --> ((2, 2, 1, 2, 2, 3, 3), (2, 2, 3, 2, 1, 2, 3))
((4, 3, 3, 2, 2, 1), ([4, 4, 3, 2, 2], 1), ([4, 3, 3, 3, 2], 1), ([7, 4, 2, 2], 1), ([7, 3, 3, 2], 1), ([6, 4, 3, 2], 3), ([5, 4, 4, 2], 1), ([5, 4, 3, 3], 1)) --> ((2, 3, 1, 2, 3, 4), (2, 3, 3, 1, 2, 4))


In [42]:
###### Example of Caterpillars [2,3,1,2,3,4] and [2,3,3,1,2,4] which equal leading, mu and nu partitions #######

print('The CSF of [2,3,1,2,3,4] is:', '\n', coefficient_printer(CSF_tree(create_caterpillar([2,3,1,2,3,4])), Partitions(15).list()))
print('----')
print('The CSF of [2,3,3,1,2,4] is', '\n', coefficient_printer(CSF_tree(create_caterpillar([2,3,3,1,2,4])), Partitions(15).list()))

The CSF of [2,3,1,2,3,4] is: 
 1*st[15] - 5*st[14, 1] + 1*st[13, 2] + 10*st[13, 1, 1] - 4*st[12, 2, 1] - 10*st[12, 1, 1, 1] + 1*st[11, 4] + 6*st[11, 2, 1, 1] + 5*st[11, 1, 1, 1, 1] + 1*st[10, 5] - 5*st[10, 4, 1] + 1*st[10, 3, 2] - 4*st[10, 2, 1, 1, 1] - 1*st[10, 1, 1, 1, 1, 1] + 1*st[9, 6] - 4*st[9, 5, 1] + 2*st[9, 4, 2] + 9*st[9, 4, 1, 1] - 3*st[9, 3, 2, 1] + 1*st[9, 2, 1, 1, 1, 1] + 1*st[8, 7] - 3*st[8, 6, 1] + 5*st[8, 5, 1, 1] + 1*st[8, 4, 3] - 5*st[8, 4, 2, 1] - 7*st[8, 4, 1, 1, 1] + 3*st[8, 3, 2, 1, 1] - 3*st[7, 7, 1] + 2*st[7, 6, 2] + 7*st[7, 6, 1, 1] + 1*st[7, 5, 3] - 4*st[7, 5, 2, 1] - 3*st[7, 5, 1, 1, 1] - 4*st[7, 4, 3, 1] + 1*st[7, 4, 2, 2] + 6*st[7, 4, 2, 1, 1] + 2*st[7, 4, 1, 1, 1, 1] + 1*st[7, 3, 3, 2] - 1*st[7, 3, 2, 2, 1] - 1*st[7, 3, 2, 1, 1, 1] - 2*st[6, 6, 2, 1] - 3*st[6, 6, 1, 1, 1] + 2*st[6, 5, 4] - 1*st[6, 5, 3, 1] + 4*st[6, 5, 2, 1, 1] + 1*st[6, 5, 1, 1, 1, 1] - 2*st[6, 4, 4, 1] + 3*st[6, 4, 3, 2] + 4*st[6, 4, 3, 1, 1] - 1*st[6, 4, 2, 2, 1] - 3*st[6, 4, 2, 1, 1, 1

In [43]:
print('These two caterpillars differ in')
compare_polynomials('1*st[15] - 5*st[14, 1] + 1*st[13, 2] + 10*st[13, 1, 1] - 4*st[12, 2, 1] - 10*st[12, 1, 1, 1] + 1*st[11, 4] + 6*st[11, 2, 1, 1] + 5*st[11, 1, 1, 1, 1] + 1*st[10, 5] - 5*st[10, 4, 1] + 1*st[10, 3, 2] - 4*st[10, 2, 1, 1, 1] - 1*st[10, 1, 1, 1, 1, 1] + 1*st[9, 6] - 4*st[9, 5, 1] + 2*st[9, 4, 2] + 9*st[9, 4, 1, 1] - 3*st[9, 3, 2, 1] + 1*st[9, 2, 1, 1, 1, 1] + 1*st[8, 7] - 3*st[8, 6, 1] + 5*st[8, 5, 1, 1] + 1*st[8, 4, 3] - 5*st[8, 4, 2, 1] - 7*st[8, 4, 1, 1, 1] + 3*st[8, 3, 2, 1, 1] - 3*st[7, 7, 1] + 2*st[7, 6, 2] + 7*st[7, 6, 1, 1] + 1*st[7, 5, 3] - 4*st[7, 5, 2, 1] - 3*st[7, 5, 1, 1, 1] - 4*st[7, 4, 3, 1] + 1*st[7, 4, 2, 2] + 6*st[7, 4, 2, 1, 1] + 2*st[7, 4, 1, 1, 1, 1] + 1*st[7, 3, 3, 2] - 1*st[7, 3, 2, 2, 1] - 1*st[7, 3, 2, 1, 1, 1] - 2*st[6, 6, 2, 1] - 3*st[6, 6, 1, 1, 1] + 2*st[6, 5, 4] - 1*st[6, 5, 3, 1] + 4*st[6, 5, 2, 1, 1] + 1*st[6, 5, 1, 1, 1, 1] - 2*st[6, 4, 4, 1] + 3*st[6, 4, 3, 2] + 4*st[6, 4, 3, 1, 1] - 1*st[6, 4, 2, 2, 1] - 3*st[6, 4, 2, 1, 1, 1] - 1*st[6, 3, 3, 2, 1] + 1*st[6, 3, 2, 2, 1, 1] - 3*st[5, 5, 4, 1] + 1*st[5, 4, 4, 2] + 4*st[5, 4, 4, 1, 1] + 1*st[5, 4, 3, 3] - 6*st[5, 4, 3, 2, 1] - 1*st[5, 4, 3, 1, 1, 1] - 1*st[4, 4, 4, 2, 1] - 1*st[4, 4, 4, 1, 1, 1] - 1*st[4, 4, 3, 3, 1] + 1*st[4, 4, 3, 2, 2] + 3*st[4, 4, 3, 2, 1, 1] + 1*st[4, 3, 3, 3, 2] - 1*st[4, 3, 3, 2, 2, 1]', '1*st[15] - 5*st[14, 1] + 1*st[13, 2] + 10*st[13, 1, 1] - 4*st[12, 2, 1] - 10*st[12, 1, 1, 1] + 1*st[11, 4] + 6*st[11, 2, 1, 1] + 5*st[11, 1, 1, 1, 1] + 1*st[10, 5] - 5*st[10, 4, 1] + 1*st[10, 3, 2] - 4*st[10, 2, 1, 1, 1] - 1*st[10, 1, 1, 1, 1, 1] + 1*st[9, 6] - 4*st[9, 5, 1] + 2*st[9, 4, 2] + 9*st[9, 4, 1, 1] - 3*st[9, 3, 2, 1] + 1*st[9, 2, 1, 1, 1, 1] + 1*st[8, 7] - 4*st[8, 6, 1] + 6*st[8, 5, 1, 1] + 1*st[8, 4, 3] - 6*st[8, 4, 2, 1] - 7*st[8, 4, 1, 1, 1] + 3*st[8, 3, 2, 1, 1] - 2*st[7, 7, 1] + 2*st[7, 6, 2] + 6*st[7, 6, 1, 1] + 1*st[7, 5, 3] - 2*st[7, 5, 2, 1] - 4*st[7, 5, 1, 1, 1] - 3*st[7, 4, 3, 1] + 1*st[7, 4, 2, 2] + 6*st[7, 4, 2, 1, 1] + 2*st[7, 4, 1, 1, 1, 1] + 1*st[7, 3, 3, 2] - 1*st[7, 3, 2, 1, 1, 1] - 3*st[6, 6, 2, 1] - 2*st[6, 6, 1, 1, 1] + 2*st[6, 5, 4] - 2*st[6, 5, 3, 1] + 4*st[6, 5, 2, 1, 1] + 1*st[6, 5, 1, 1, 1, 1] - 2*st[6, 4, 4, 1] + 3*st[6, 4, 3, 2] + 3*st[6, 4, 3, 1, 1] - 2*st[6, 4, 2, 2, 1] - 2*st[6, 4, 2, 1, 1, 1] - 2*st[6, 3, 3, 2, 1] - 3*st[5, 5, 4, 1] + 1*st[5, 5, 3, 1, 1] - 1*st[5, 5, 2, 1, 1, 1] + 1*st[5, 4, 4, 2] + 4*st[5, 4, 4, 1, 1] + 1*st[5, 4, 3, 3] - 5*st[5, 4, 3, 2, 1] - 1*st[5, 4, 3, 1, 1, 1] + 1*st[5, 4, 2, 2, 1, 1] + 1*st[5, 3, 3, 2, 1, 1] - 1*st[4, 4, 4, 2, 1] - 1*st[4, 4, 4, 1, 1, 1] - 1*st[4, 4, 3, 3, 1] + 1*st[4, 4, 3, 2, 2] + 2*st[4, 4, 3, 2, 1, 1] + 1*st[4, 3, 3, 3, 2] - 1*st[4, 3, 3, 2, 2, 1]')

These two caterpillars differ in
| Term                     | Coefficient 1 | Coefficient 2 |
|--------------------------|---------------|---------------|
| st[4, 4, 3, 2, 1, 1]     | 3             | 2             |
| st[5, 3, 3, 2, 1, 1]     | 0             | 1             |
| st[5, 4, 2, 2, 1, 1]     | 0             | 1             |
| st[5, 4, 3, 2, 1]        | 6             | 5             |
| st[5, 5, 2, 1, 1, 1]     | 0             | 1             |
| st[5, 5, 3, 1, 1]        | 0             | 1             |
| st[6, 3, 2, 2, 1, 1]     | 1             | 0             |
| st[6, 3, 3, 2, 1]        | 1             | 2             |
| st[6, 4, 2, 1, 1, 1]     | 3             | 2             |
| st[6, 4, 2, 2, 1]        | 1             | 2             |
| st[6, 4, 3, 1, 1]        | 4             | 3             |
| st[6, 5, 3, 1]           | 1             | 2             |
| st[6, 6, 1, 1, 1]        | 3             | 2             |
| st[6, 6, 2, 1]           | 2             | 3      

In [46]:
############# Example 2: Caterpillars with equal LC polynomial, so equal mu and nu partitions in particular ######

print('CSF of [2,2,3,2,1,2,3]', '\n', coefficient_printer(CSF_tree(create_caterpillar([2,2,3,2,1,2,3])), Partitions(15).list()))
print('---------')
print('CSF of [3,3,2,2,1,2,2]', '\n',coefficient_printer(CSF_tree(create_caterpillar([3,3,2,2,1,2,2])), Partitions(15).list()))

CSF of [2,2,3,2,1,2,3] 
 1*st[15] - 6*st[14, 1] + 1*st[13, 2] + 15*st[13, 1, 1] + 1*st[12, 3] - 5*st[12, 2, 1] - 20*st[12, 1, 1, 1] + 1*st[11, 4] - 6*st[11, 3, 1] + 1*st[11, 2, 2] + 10*st[11, 2, 1, 1] + 15*st[11, 1, 1, 1, 1] + 1*st[10, 5] - 5*st[10, 4, 1] + 2*st[10, 3, 2] + 14*st[10, 3, 1, 1] - 4*st[10, 2, 2, 1] - 10*st[10, 2, 1, 1, 1] - 6*st[10, 1, 1, 1, 1, 1] + 1*st[9, 6] - 5*st[9, 5, 1] + 10*st[9, 4, 1, 1] + 1*st[9, 3, 3] - 8*st[9, 3, 2, 1] - 16*st[9, 3, 1, 1, 1] + 6*st[9, 2, 2, 1, 1] + 5*st[9, 2, 1, 1, 1, 1] + 1*st[9, 1, 1, 1, 1, 1, 1] + 1*st[8, 7] - 5*st[8, 6, 1] + 2*st[8, 5, 2] + 10*st[8, 5, 1, 1] + 2*st[8, 4, 3] - 2*st[8, 4, 2, 1] - 10*st[8, 4, 1, 1, 1] - 5*st[8, 3, 3, 1] + 3*st[8, 3, 2, 2] + 12*st[8, 3, 2, 1, 1] + 9*st[8, 3, 1, 1, 1, 1] - 4*st[8, 2, 2, 1, 1, 1] - 1*st[8, 2, 1, 1, 1, 1, 1] - 3*st[7, 7, 1] + 2*st[7, 6, 2] + 12*st[7, 6, 1, 1] + 2*st[7, 5, 3] - 9*st[7, 5, 2, 1] - 11*st[7, 5, 1, 1, 1] - 9*st[7, 4, 3, 1] + 7*st[7, 4, 2, 1, 1] + 5*st[7, 4, 1, 1, 1, 1] + 3*st[7, 3, 3, 

In [48]:
print('These two differ in')
compare_polynomials('1*st[15] - 6*st[14, 1] + 1*st[13, 2] + 15*st[13, 1, 1] + 1*st[12, 3] - 5*st[12, 2, 1] - 20*st[12, 1, 1, 1] + 1*st[11, 4] - 6*st[11, 3, 1] + 1*st[11, 2, 2] + 10*st[11, 2, 1, 1] + 15*st[11, 1, 1, 1, 1] + 1*st[10, 5] - 5*st[10, 4, 1] + 2*st[10, 3, 2] + 14*st[10, 3, 1, 1] - 4*st[10, 2, 2, 1] - 10*st[10, 2, 1, 1, 1] - 6*st[10, 1, 1, 1, 1, 1] + 1*st[9, 6] - 5*st[9, 5, 1] + 10*st[9, 4, 1, 1] + 1*st[9, 3, 3] - 8*st[9, 3, 2, 1] - 16*st[9, 3, 1, 1, 1] + 6*st[9, 2, 2, 1, 1] + 5*st[9, 2, 1, 1, 1, 1] + 1*st[9, 1, 1, 1, 1, 1, 1] + 1*st[8, 7] - 5*st[8, 6, 1] + 2*st[8, 5, 2] + 10*st[8, 5, 1, 1] + 2*st[8, 4, 3] - 2*st[8, 4, 2, 1] - 10*st[8, 4, 1, 1, 1] - 5*st[8, 3, 3, 1] + 3*st[8, 3, 2, 2] + 12*st[8, 3, 2, 1, 1] + 9*st[8, 3, 1, 1, 1, 1] - 4*st[8, 2, 2, 1, 1, 1] - 1*st[8, 2, 1, 1, 1, 1, 1] - 3*st[7, 7, 1] + 2*st[7, 6, 2] + 12*st[7, 6, 1, 1] + 2*st[7, 5, 3] - 9*st[7, 5, 2, 1] - 11*st[7, 5, 1, 1, 1] - 9*st[7, 4, 3, 1] + 7*st[7, 4, 2, 1, 1] + 5*st[7, 4, 1, 1, 1, 1] + 3*st[7, 3, 3, 2] + 10*st[7, 3, 3, 1, 1] - 10*st[7, 3, 2, 2, 1] - 8*st[7, 3, 2, 1, 1, 1] - 2*st[7, 3, 1, 1, 1, 1, 1] + 1*st[7, 2, 2, 1, 1, 1, 1] - 4*st[6, 6, 2, 1] - 7*st[6, 6, 1, 1, 1] + 2*st[6, 5, 4] - 6*st[6, 5, 3, 1] + 3*st[6, 5, 2, 2] + 14*st[6, 5, 2, 1, 1] + 7*st[6, 5, 1, 1, 1, 1] - 2*st[6, 4, 4, 1] + 2*st[6, 4, 3, 2] + 14*st[6, 4, 3, 1, 1] - 3*st[6, 4, 2, 2, 1] - 8*st[6, 4, 2, 1, 1, 1] - 1*st[6, 4, 1, 1, 1, 1, 1] - 8*st[6, 3, 3, 2, 1] - 9*st[6, 3, 3, 1, 1, 1] + 2*st[6, 3, 2, 2, 2] + 11*st[6, 3, 2, 2, 1, 1] + 2*st[6, 3, 2, 1, 1, 1, 1] - 3*st[5, 5, 4, 1] + 2*st[5, 5, 3, 2] + 5*st[5, 5, 3, 1, 1] - 5*st[5, 5, 2, 2, 1] - 5*st[5, 5, 2, 1, 1, 1] - 1*st[5, 5, 1, 1, 1, 1, 1] + 4*st[5, 4, 4, 1, 1] + 3*st[5, 4, 3, 3] - 9*st[5, 4, 3, 2, 1] - 9*st[5, 4, 3, 1, 1, 1] + 7*st[5, 4, 2, 2, 1, 1] + 3*st[5, 4, 2, 1, 1, 1, 1] - 3*st[5, 3, 3, 3, 1] + 5*st[5, 3, 3, 2, 2] + 8*st[5, 3, 3, 2, 1, 1] + 3*st[5, 3, 3, 1, 1, 1, 1] - 5*st[5, 3, 2, 2, 2, 1] - 4*st[5, 3, 2, 2, 1, 1, 1] - 1*st[4, 4, 4, 1, 1, 1] - 4*st[4, 4, 3, 3, 1] + 5*st[4, 4, 3, 2, 1, 1] + 1*st[4, 4, 3, 1, 1, 1, 1] - 2*st[4, 4, 2, 2, 1, 1, 1] + 2*st[4, 3, 3, 3, 2] + 5*st[4, 3, 3, 3, 1, 1] - 7*st[4, 3, 3, 2, 2, 1] - 3*st[4, 3, 3, 2, 1, 1, 1] + 3*st[4, 3, 2, 2, 2, 1, 1] - 2*st[3, 3, 3, 3, 2, 1] - 1*st[3, 3, 3, 3, 1, 1, 1] + 2*st[3, 3, 3, 2, 2, 2] + 2*st[3, 3, 3, 2, 2, 1, 1] - 1*st[3, 3, 2, 2, 2, 2, 1]', '1*st[15] - 6*st[14, 1] + 1*st[13, 2] + 15*st[13, 1, 1] + 1*st[12, 3] - 5*st[12, 2, 1] - 20*st[12, 1, 1, 1] + 1*st[11, 4] - 6*st[11, 3, 1] + 1*st[11, 2, 2] + 10*st[11, 2, 1, 1] + 15*st[11, 1, 1, 1, 1] + 1*st[10, 5] - 5*st[10, 4, 1] + 2*st[10, 3, 2] + 14*st[10, 3, 1, 1] - 4*st[10, 2, 2, 1] - 10*st[10, 2, 1, 1, 1] - 6*st[10, 1, 1, 1, 1, 1] + 1*st[9, 6] - 4*st[9, 5, 1] + 9*st[9, 4, 1, 1] + 1*st[9, 3, 3] - 7*st[9, 3, 2, 1] - 16*st[9, 3, 1, 1, 1] + 6*st[9, 2, 2, 1, 1] + 5*st[9, 2, 1, 1, 1, 1] + 1*st[9, 1, 1, 1, 1, 1, 1] + 1*st[8, 7] - 7*st[8, 6, 1] + 2*st[8, 5, 2] + 10*st[8, 5, 1, 1] + 2*st[8, 4, 3] - 4*st[8, 4, 2, 1] - 8*st[8, 4, 1, 1, 1] - 6*st[8, 3, 3, 1] + 3*st[8, 3, 2, 2] + 11*st[8, 3, 2, 1, 1] + 9*st[8, 3, 1, 1, 1, 1] - 1*st[8, 2, 2, 2, 1] - 4*st[8, 2, 2, 1, 1, 1] - 1*st[8, 2, 1, 1, 1, 1, 1] - 2*st[7, 7, 1] + 2*st[7, 6, 2] + 13*st[7, 6, 1, 1] + 2*st[7, 5, 3] - 6*st[7, 5, 2, 1] - 13*st[7, 5, 1, 1, 1] - 7*st[7, 4, 3, 1] + 8*st[7, 4, 2, 1, 1] + 4*st[7, 4, 1, 1, 1, 1] + 3*st[7, 3, 3, 2] + 11*st[7, 3, 3, 1, 1] - 7*st[7, 3, 2, 2, 1] - 9*st[7, 3, 2, 1, 1, 1] - 2*st[7, 3, 1, 1, 1, 1, 1] + 2*st[7, 2, 2, 2, 1, 1] + 1*st[7, 2, 2, 1, 1, 1, 1] - 6*st[6, 6, 2, 1] - 7*st[6, 6, 1, 1, 1] + 2*st[6, 5, 4] - 7*st[6, 5, 3, 1] + 3*st[6, 5, 2, 2] + 14*st[6, 5, 2, 1, 1] + 8*st[6, 5, 1, 1, 1, 1] - 3*st[6, 4, 4, 1] + 2*st[6, 4, 3, 2] + 13*st[6, 4, 3, 1, 1] - 6*st[6, 4, 2, 2, 1] - 6*st[6, 4, 2, 1, 1, 1] - 1*st[6, 4, 1, 1, 1, 1, 1] - 10*st[6, 3, 3, 2, 1] - 9*st[6, 3, 3, 1, 1, 1] + 2*st[6, 3, 2, 2, 2] + 8*st[6, 3, 2, 2, 1, 1] + 3*st[6, 3, 2, 1, 1, 1, 1] - 1*st[6, 2, 2, 2, 2, 1] - 1*st[6, 2, 2, 2, 1, 1, 1] - 2*st[5, 5, 4, 1] + 2*st[5, 5, 3, 2] + 6*st[5, 5, 3, 1, 1] - 3*st[5, 5, 2, 2, 1] - 6*st[5, 5, 2, 1, 1, 1] - 1*st[5, 5, 1, 1, 1, 1, 1] + 3*st[5, 4, 4, 1, 1] + 3*st[5, 4, 3, 3] - 8*st[5, 4, 3, 2, 1] - 10*st[5, 4, 3, 1, 1, 1] + 6*st[5, 4, 2, 2, 1, 1] + 2*st[5, 4, 2, 1, 1, 1, 1] - 3*st[5, 3, 3, 3, 1] + 5*st[5, 3, 3, 2, 2] + 11*st[5, 3, 3, 2, 1, 1] + 3*st[5, 3, 3, 1, 1, 1, 1] - 3*st[5, 3, 2, 2, 2, 1] - 4*st[5, 3, 2, 2, 1, 1, 1] + 1*st[5, 2, 2, 2, 2, 1, 1] - 4*st[4, 4, 3, 3, 1] + 4*st[4, 4, 3, 2, 1, 1] + 1*st[4, 4, 3, 1, 1, 1, 1] + 2*st[4, 3, 3, 3, 2] + 5*st[4, 3, 3, 3, 1, 1] - 8*st[4, 3, 3, 2, 2, 1] - 4*st[4, 3, 3, 2, 1, 1, 1] + 1*st[4, 3, 2, 2, 2, 1, 1] - 2*st[3, 3, 3, 3, 2, 1] - 1*st[3, 3, 3, 3, 1, 1, 1] + 2*st[3, 3, 3, 2, 2, 2] + 3*st[3, 3, 3, 2, 2, 1, 1] - 1*st[3, 3, 2, 2, 2, 2, 1]')

These two differ in
| Term                     | Coefficient 1 | Coefficient 2 |
|--------------------------|---------------|---------------|
| st[3, 3, 3, 2, 2, 1, 1]  | 2             | 3             |
| st[4, 3, 2, 2, 2, 1, 1]  | 3             | 1             |
| st[4, 3, 3, 2, 1, 1, 1]  | 3             | 4             |
| st[4, 3, 3, 2, 2, 1]     | 7             | 8             |
| st[4, 4, 2, 2, 1, 1, 1]  | 2             | 0             |
| st[4, 4, 3, 2, 1, 1]     | 5             | 4             |
| st[4, 4, 4, 1, 1, 1]     | 1             | 0             |
| st[5, 2, 2, 2, 2, 1, 1]  | 0             | 1             |
| st[5, 3, 2, 2, 2, 1]     | 5             | 3             |
| st[5, 3, 3, 2, 1, 1]     | 8             | 11            |
| st[5, 4, 2, 1, 1, 1, 1]  | 3             | 2             |
| st[5, 4, 2, 2, 1, 1]     | 7             | 6             |
| st[5, 4, 3, 1, 1, 1]     | 9             | 10            |
| st[5, 4, 3, 2, 1]        | 9             | 8             |
| st

In [40]:
###### In progress - Mario testing ideas

def mu_nutilde_partitions(CSFvector,n):
    partitions = Partitions(n).list()
    leading = get_leading_partition(CSFvector, n)
    mu_nu_partitions = []
    
    for i in range(len(partitions)):
        lbda = partitions[i]
        if len(lbda) == len(leading)-1 and count_ones(lbda)==0 and CSFvector[i]!=0:
            lst = [lbda, CSFvector[i]]
            tpl = tuple(lst)
            mu_nu_partitions.append(tpl)
    
    for i in range(len(partitions)):
        lbda = partitions[i]
        if len(lbda) == len(leading)-1 and count_ones(lbda)==1 and CSFvector[i]!=0:
            lst = [lbda, CSFvector[i]]
            tpl = tuple(lst)
            mu_nu_partitions.append(tpl)
    return mu_nu_partitions

# Output: a map that sends a caterpillar --> a tuple containing its leading partition, and its mu and nu tilde
#         partitions as defined in the function above
def caterpillars_leading_nutilde_map(n):
    map = {}
    for comp in generate_caterpillars(n):
        if count_ones(comp)==1:
            C=create_caterpillar(comp)
            C=C.copy()
            lst = list(mu_nutilde_partitions(CSF_tree(C), n))
            lst.insert(0, tuple(sorted(comp, reverse=True)))
            tpl = tuple(lst)
            map[tuple(comp)]=tpl
    return map

In [41]:
mu_nutilde_partitions(CSF_tree(create_caterpillar([2,3,1,2,3,4])), 15)

[([4, 4, 3, 2, 2], 1),
 ([4, 3, 3, 3, 2], 1),
 ([7, 3, 2, 2, 1], -1),
 ([6, 4, 2, 2, 1], -1),
 ([6, 3, 3, 2, 1], -1),
 ([5, 4, 3, 2, 1], -6),
 ([4, 4, 4, 2, 1], -1),
 ([4, 4, 3, 3, 1], -1)]

In [42]:
mu_nutilde_partitions(CSF_tree(create_caterpillar([2,3,3,1,2,4])), 15)

[([4, 4, 3, 2, 2], 1),
 ([4, 3, 3, 3, 2], 1),
 ([6, 4, 2, 2, 1], -2),
 ([6, 3, 3, 2, 1], -2),
 ([5, 4, 3, 2, 1], -5),
 ([4, 4, 4, 2, 1], -1),
 ([4, 4, 3, 3, 1], -1)]

In [43]:
n=12

mp = caterpillars_leading_nutilde_map(n)

flipped = {}

for key, value in mp.items():
    tplkey = tuple(key)
    tplvalue=tuple(value)
    
    if tplvalue not in flipped.keys():
        flipped[tplvalue] = tuple([tplkey])

    else:
        lst = list(flipped[tplvalue])
        lst.append(tplkey)
        tpl = tuple(lst)
        flipped[tplvalue]=tpl

for key in flipped.keys():
    if len(flipped[key])>1:
        for i in range(len(flipped[key])-1):
            if not caterpillar_edges(flipped[key][i]) == caterpillar_edges(flipped[key][i+1]):
                print(key, '-->', flipped[key])
                break